# Perpetrator PCA train-only — reduced deterministic grid

Versión para probar reproducibilidad sin lanzar el grid completo.

Características:
- `output_dir='./content/perpetrator'` mantenido.
- PCA train-only: split primero; scaler/PCA fit solo en train; test transformado.
- Sin `mixed_bfloat16`; usa `float32`.
- Semillas fijadas para Python/NumPy/TensorFlow.
- `shuffle` determinista con `reshuffle_each_iteration=False`.
- Dropout e inicializadores con seed.
- Grid reducido y multi-seed: no prueba 330 candidatos, solo candidatos cercanos a los que dieron sensibilidad alta.
- No se queda con el primer candidato que cumple; selecciona por criterio estable.


In [ ]:

# ==============================
# CONFIGURACIÓN
# ==============================
import os

# Mantener como pidió Ángel: siempre guarda lo último aquí.
OUTPUT_DIR = './content/perpetrator'

# Reproducibilidad / velocidad
RANDOM_STATE = 42
FORCE_CPU_FOR_REPRODUCIBILITY = True   # Si tarda demasiado, poner False y reiniciar kernel.
ENABLE_STRICT_OP_DETERMINISM = False   # True puede ser MUCHO más lento. Primero probar False.

# Entrenamiento
BATCH_SIZE = 128
PCA_THRESHOLD = 0.99
THRESHOLD = 0.50
EPOCHS = 120
PATIENCE = 8
LEARNING_RATE = 1e-3

# Criterio de selección: priorizar screening sensible, pero no aceptar especificidad casi cero.
TARGET_RECALL_POS = 0.90
MIN_RECALL_NEG = 0.20
FALLBACK_RECALL_POS = 0.85

# Seeds de entrenamiento por arquitectura. Dos seeds suelen ser suficiente para comprobar estabilidad sin alargar demasiado.
TRAINING_SEEDS = [42, 123]

if FORCE_CPU_FOR_REPRODUCIBILITY:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)

# ==============================
# IMPORTS
# ==============================
import json
import random
import itertools
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, Callback

# Semillas globales
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

# Sin mixed precision para evitar variaciones y comportamiento raro.
tf.keras.mixed_precision.set_global_policy('float32')

if ENABLE_STRICT_OP_DETERMINISM:
    try:
        tf.config.experimental.enable_op_determinism()
        print('TensorFlow op determinism: ON')
    except Exception as e:
        print('No se pudo activar enable_op_determinism:', repr(e))
else:
    print('TensorFlow op determinism: OFF (más rápido)')

print('TF version:', tf.__version__)
print('GPUs visibles:', tf.config.list_physical_devices('GPU'))
print('Política precisión:', tf.keras.mixed_precision.global_policy())

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ==============================
# UTILIDADES
# ==============================
def read_csv_fallback(primary, fallback):
    """Lee primary si existe; si no, fallback. Útil para ejecutar desde carpetas distintas."""
    if os.path.exists(primary):
        return pd.read_csv(primary)
    if os.path.exists(fallback):
        return pd.read_csv(fallback)
    raise FileNotFoundError(f'No encuentro ni {primary!r} ni {fallback!r}')


def pca_variance_df(pca_model):
    explained = pca_model.explained_variance_ratio_
    return pd.DataFrame({
        'PC': [f'PC{i+1}' for i in range(len(explained))],
        'Explained_Variance': explained,
        'Cumulative_Variance': np.cumsum(explained),
    })


def select_n_components(df_var, threshold):
    mask = df_var['Cumulative_Variance'] >= threshold
    if not mask.any():
        return len(df_var)
    return int(mask.idxmax() + 1)


class OverfitStopping(Callback):
    def __init__(self, threshold=0.10):
        super().__init__()
        self.threshold = threshold
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        tr = logs.get('recall')
        vr = logs.get('val_recall')
        if tr is not None and vr is not None and (tr - vr) > self.threshold:
            self.model.stop_training = True


def make_dataset(X, y, batch_size, seed=None, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X.astype('float32'), y.astype('float32')))
    if training:
        ds = ds.shuffle(buffer_size=len(y), seed=seed, reshuffle_each_iteration=False)
    ds = ds.batch(batch_size)
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)
    return ds


def build_model(input_dim, params, seed):
    # Reseteo fuerte de semilla por candidato.
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    u1, a1, d1, u2, a2, d2 = params
    k1 = tf.keras.initializers.GlorotUniform(seed=seed + 1)
    k2 = tf.keras.initializers.GlorotUniform(seed=seed + 2)
    k3 = tf.keras.initializers.GlorotUniform(seed=seed + 3)

    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(u1, activation=a1, kernel_initializer=k1, bias_initializer='zeros'),
        layers.Dropout(d1, seed=seed + 11),
        layers.Dense(u2, activation=a2, kernel_initializer=k2, bias_initializer='zeros'),
        layers.Dropout(d2, seed=seed + 12),
        layers.Dense(1, activation='sigmoid', kernel_initializer=k3, bias_initializer='zeros', dtype='float32'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        ],
    )
    return model


def metric_row(y_true, y_prob, params, seed, epochs_ran):
    y_pred = (y_prob >= THRESHOLD).astype(int)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall0 = report.get('0', {}).get('recall', 0.0)
    recall1 = report.get('1', {}).get('recall', 0.0)
    precision1 = report.get('1', {}).get('precision', 0.0)
    f1_1 = report.get('1', {}).get('f1-score', 0.0)
    accuracy = report.get('accuracy', 0.0)
    bal_acc = (recall0 + recall1) / 2

    # Tier estable: primero intentamos el objetivo del paper; luego fallback razonable.
    if recall1 >= TARGET_RECALL_POS and recall0 >= MIN_RECALL_NEG:
        tier = 3
    elif recall1 >= FALLBACK_RECALL_POS and recall0 >= MIN_RECALL_NEG:
        tier = 2
    elif recall1 >= TARGET_RECALL_POS:
        tier = 1
    else:
        tier = 0

    # Score lexicográfico: no escoge “el primero”, sino el mejor según criterio fijo.
    selection_score = (tier, bal_acc, recall0, precision1, accuracy, recall1)

    u1, a1, d1, u2, a2, d2 = params
    return {
        'seed': seed,
        'u1': u1, 'a1': a1, 'd1': d1,
        'u2': u2, 'a2': a2, 'd2': d2,
        'threshold': THRESHOLD,
        'recall_0': recall0,
        'recall_1': recall1,
        'specificity': recall0,
        'precision_ppv': precision1,
        'f1_positive': f1_1,
        'accuracy': accuracy,
        'balanced_accuracy': bal_acc,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'epochs_ran': int(epochs_ran),
        'tier': int(tier),
        'selection_score': selection_score,
    }


# ==============================
# CARGA Y PREPARACIÓN DE DATOS
# ==============================
feat_df = read_csv_fallback('./../data/lista_global_vars.csv', './lista_global_vars.csv')
target_df = read_csv_fallback('./../data/target_col.csv', './target_col.csv').fillna(0)

# Mismo merge/filtro que el notebook original.
df_merged = feat_df.join(target_df, how='inner')
df_merged = df_merged[
    ~((df_merged['GENERO_BIN_2'] == 1) | (df_merged['ORIENTSEX.BN_3'] == 1))
].drop(columns=['GENERO_BIN_2', 'ORIENTSEX.BN_3']).reset_index(drop=True)

cols_to_drop = [
    'VÍCTIMA', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION', 'POLIPERPETRACION',
    'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP', 'V.O',
    'P.SUM.TOTAL', 'V.SUM.TOTAL'
]

df_merged_perpetrador = df_merged.drop(columns=cols_to_drop)
df_features = df_merged_perpetrador.drop(columns=['PERPETRADOR'])
y_series = df_merged_perpetrador['PERPETRADOR'].astype(int)

# Guardar dataset base de control.
df_features.to_csv(os.path.join(OUTPUT_DIR, 'df_perpetrador_feat.csv'), index=False)
y_series.to_csv(os.path.join(OUTPUT_DIR, 'df_perpretador_target.csv'), index=False)

print('Analytical n:', len(df_features))
print('Target counts:', Counter(y_series))

# ==============================
# SPLIT PRIMERO; SCALER/PCA SOLO TRAIN
# ==============================
all_idx = df_features.index.to_numpy()
train_idx, val_idx = train_test_split(
    all_idx,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_series,
)

with open(os.path.join(OUTPUT_DIR, 'splits_indices.json'), 'w') as f:
    json.dump({'train': train_idx.tolist(), 'val': val_idx.tolist(), 'pca_scope': 'train_only'}, f, indent=2)

X_train_raw = df_features.loc[train_idx].copy()
X_val_raw = df_features.loc[val_idx].copy()
y_train = y_series.loc[train_idx].values.astype(int)
y_val = y_series.loc[val_idx].values.astype(int)

scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

means = pd.Series(X_train_scaled.mean(axis=0), index=X_train_raw.columns, name='mean_train_scaled')
X_train_centered = X_train_scaled - means.values
X_val_centered = X_val_scaled - means.values

pca = PCA()
X_train_pca_full = pca.fit_transform(X_train_centered)
X_val_pca_full = pca.transform(X_val_centered)

df_variance = pca_variance_df(pca)
n_components = select_n_components(df_variance, PCA_THRESHOLD)

X_train = X_train_pca_full[:, :n_components].astype('float32')
X_val = X_val_pca_full[:, :n_components].astype('float32')
pc_cols = [f'PC{i+1}' for i in range(n_components)]

print(f'PCA components selected @ {PCA_THRESHOLD:.2f}:', n_components)

# Guardar PCA/scaler/datos transformados.
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler_minmax.pkl'))
joblib.dump(pca, os.path.join(OUTPUT_DIR, 'modelo_pca.pkl'))
means.to_csv(os.path.join(OUTPUT_DIR, 'medias_escalado.csv'), header=True)
df_variance.to_csv(os.path.join(OUTPUT_DIR, 'df_PCA_variance_trainonly.csv'), index=False)
pd.DataFrame(X_train, columns=pc_cols).to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
pd.DataFrame(X_val, columns=pc_cols).to_csv(os.path.join(OUTPUT_DIR, 'X_val.csv'), index=False)
pd.Series(y_train, name='PERPETRADOR').to_csv(os.path.join(OUTPUT_DIR, 'y_train.csv'), index=False)
pd.Series(y_val, name='PERPETRADOR').to_csv(os.path.join(OUTPUT_DIR, 'y_val.csv'), index=False)
pd.DataFrame(np.vstack([X_train, X_val]), columns=pc_cols).to_csv(
    os.path.join(OUTPUT_DIR, f'df_PCA_{int(PCA_THRESHOLD*100)}_trainonly_stacked.csv'),
    index=False,
)

# Class weight igual que original: clase 1 ponderada por ratio negativos/positivos.
counts = Counter(y_train)
class_weight = {0: 1.0, 1: counts[0] / counts[1]}
print('class_weight:', class_weight)

# ==============================
# GRID REDUCIDO DETERMINISTA
# ==============================
# Candidatos tomados de las zonas que antes daban recall alto y/o mejor balanced accuracy.
# Formato: (u1, a1, d1, u2, a2, d2)
CANDIDATES = [
    (64, 'linear',  0.1, 16, 'sigmoid', 0.05),
    (64, 'sigmoid', 0.3, 16, 'linear',  0.00),
    (64, 'tanh',    0.1, 16, 'sigmoid', 0.00),
    (64, 'tanh',    0.1, 16, 'sigmoid', 0.05),
    (64, 'linear',  0.3,  4, 'relu',    0.10),
    (64, 'linear',  0.2,  4, 'relu',    0.10),
    (64, 'relu',    0.2,  4, 'relu',    0.10),
    (64, 'linear',  0.1,  8, 'sigmoid', 0.00),
    (64, 'relu',    0.3, 16, 'sigmoid', 0.00),
    (64, 'tanh',    0.1,  8, 'sigmoid', 0.00),
    (64, 'relu',    0.1,  8, 'relu',    0.05),
    (64, 'linear',  0.3,  4, 'relu',    0.05),
    (64, 'linear',  0.1, 16, 'relu',    0.10),
    (64, 'relu',    0.2,  8, 'sigmoid', 0.05),
    (64, 'relu',    0.3, 16, 'linear',  0.05),
    (64, 'relu',    0.3, 16, 'linear',  0.00),
]

# El número total real será len(CANDIDATES) * len(TRAINING_SEEDS).
print('N candidates:', len(CANDIDATES), 'x seeds:', len(TRAINING_SEEDS), '=', len(CANDIDATES) * len(TRAINING_SEEDS))

results = []
best = None
best_report = None
best_pred = None
best_probs = None
best_params = None
best_seed = None

for cand_i, params in enumerate(CANDIDATES, start=1):
    for seed_i, train_seed in enumerate(TRAINING_SEEDS, start=1):
        candidate_seed = int(train_seed + cand_i * 1000)
        print(f'\n[{cand_i:02d}/{len(CANDIDATES)} seed {seed_i}/{len(TRAINING_SEEDS)}] params={params}, seed={candidate_seed}')

        tf.keras.backend.clear_session()
        train_ds = make_dataset(X_train, y_train, BATCH_SIZE, seed=candidate_seed, training=True)
        val_ds = make_dataset(X_val, y_val, BATCH_SIZE, seed=None, training=False)
        model = build_model(X_train.shape[1], params, candidate_seed)

        callbacks = [
            OverfitStopping(0.10),
            EarlyStopping(monitor='val_recall', mode='max', patience=PATIENCE, restore_best_weights=True),
        ]

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=0,
            shuffle=False,
        )

        y_prob = model.predict(val_ds, verbose=0).reshape(-1)
        row = metric_row(y_val, y_prob, params, candidate_seed, epochs_ran=len(history.history.get('loss', [])))
        results.append(row)

        print({k: row[k] for k in ['recall_1', 'recall_0', 'precision_ppv', 'balanced_accuracy', 'accuracy', 'TP', 'FP', 'TN', 'FN', 'tier', 'epochs_ran']})

        if best is None or row['selection_score'] > best['selection_score']:
            best = row
            best_params = params
            best_seed = candidate_seed
            best_probs = y_prob.copy()
            best_pred = (y_prob >= THRESHOLD).astype(int)
            best_report = classification_report(y_val, best_pred, output_dict=True, zero_division=0)
            model.save(os.path.join(OUTPUT_DIR, 'best_model.h5'))
            print('  -> Nuevo BEST guardado')

# Guardar resultados del grid reducido.
df_results = pd.DataFrame(results).drop(columns=['selection_score'])
df_results.to_csv(os.path.join(OUTPUT_DIR, 'gridsearch_results.csv'), index=False)

# Guardar best report.
with open(os.path.join(OUTPUT_DIR, 'best_report.json'), 'w') as f:
    json.dump(best_report, f, indent=2)

# Guardar config completa.
config = {
    'random_state': RANDOM_STATE,
    'force_cpu_for_reproducibility': FORCE_CPU_FOR_REPRODUCIBILITY,
    'enable_strict_op_determinism': ENABLE_STRICT_OP_DETERMINISM,
    'threshold': THRESHOLD,
    'pca_threshold': PCA_THRESHOLD,
    'n_components': int(n_components),
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'learning_rate': LEARNING_RATE,
    'class_weight': class_weight,
    'training_seeds': TRAINING_SEEDS,
    'selection_rule': {
        'tier_3': f'recall_1 >= {TARGET_RECALL_POS} and recall_0 >= {MIN_RECALL_NEG}',
        'tier_2': f'recall_1 >= {FALLBACK_RECALL_POS} and recall_0 >= {MIN_RECALL_NEG}',
        'tier_1': f'recall_1 >= {TARGET_RECALL_POS}',
        'score': '(tier, balanced_accuracy, recall_0, precision_ppv, accuracy, recall_1)',
    },
    'best': {k: v for k, v in best.items() if k != 'selection_score'},
    'best_params': {
        'u1': best_params[0], 'a1': best_params[1], 'd1': best_params[2],
        'u2': best_params[3], 'a2': best_params[4], 'd2': best_params[5],
        'seed': best_seed,
    },
}
with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

# Guardar predicciones del mejor modelo.
df_pred = pd.DataFrame({
    'idx_original': val_idx,
    'y_true_perp': y_val.astype(int),
    'y_pred_perp': best_pred.astype(int),
    'y_prob_perp': best_probs.astype(float),
}).sort_values('idx_original').reset_index(drop=True)
df_pred.to_csv(os.path.join(OUTPUT_DIR, 'predictions_with_probs.csv'), index=False)

# Guardar métricas finales compactas.
metrics_final = pd.DataFrame([{k: best[k] for k in [
    'threshold', 'accuracy', 'balanced_accuracy', 'precision_ppv', 'recall_1',
    'specificity', 'f1_positive', 'TP', 'FP', 'TN', 'FN'
]}]).rename(columns={'recall_1': 'recall_sensitivity'})
# NPV manual
tn, fp, fn, tp = best['TN'], best['FP'], best['FN'], best['TP']
metrics_final['npv'] = tn / (tn + fn) if (tn + fn) > 0 else np.nan
metrics_final['support'] = len(y_val)
metrics_final.to_csv(os.path.join(OUTPUT_DIR, 'metrics_final.csv'), index=False)

print('\n=== BEST SELECTED ===')
print(pd.Series({k: v for k, v in best.items() if k != 'selection_score'}))
print('\n=== FINAL REPORT — PERPETRATOR CSV ===')
print(classification_report(y_val, best_pred, digits=3, zero_division=0))

print('\nGuardado en:', OUTPUT_DIR)
print('Archivos principales: predictions_with_probs.csv, best_report.json, gridsearch_results.csv, metrics_final.csv, config.json, best_model.h5')


In [ ]:

# CHECK opcional: ejecutar SOLO después de que haya terminado la celda anterior.
import pandas as pd
from sklearn.metrics import classification_report

pred_path = './content/perpetrator/predictions_with_probs.csv'
data_set = pd.read_csv(pred_path)

print('=== CHECK predictions_with_probs.csv ===')
print(data_set.head())
print(data_set.shape)
print(list(data_set.columns))
print(classification_report(data_set['y_true_perp'], data_set['y_pred_perp'], digits=3, zero_division=0))

print('\n=== BEST CONFIG ===')
print(pd.read_json('./content/perpetrator/config.json'))
